<a href="https://colab.research.google.com/drive/1ZQG9Hnv1caDkcsmoCAf8_ynpqs3IQmST?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

### Self-Refine: Iterative Refinement with Self-Feedback

In [1]:
!pip install -qU google-generativeai


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import google.generativeai as genai
import getpass

Get free-tier Google's Gemini API Key here: https://aistudio.google.com/app/apikey

In [3]:
# Prefer an environment variable, fall back to prompting.
# The prompt alone meant these notebooks could not run non-interactively
# (nbconvert, papermill, CI) and made you retype the key once per notebook.
import os
API_KEY = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    API_KEY = getpass.getpass("Enter your Google API key: ")

In [4]:
genai.configure(api_key=API_KEY)

In [5]:
class SelfRefineAgent:
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-flash-latest")
        self.history = []

    def generate(self, task):
        """Generate initial output"""
        prompt = f"""Generate a response for this task:

        Task: {task}

        Response:"""

        response = self.model.generate_content(prompt).text
        return response.strip()

    def critique(self, task, output):
        """Self-critique: Identify issues and areas for improvement"""
        prompt = f"""Task: {task}

        Current Output:
        {output}

        Critique this output. Identify:
        1. What's good about it
        2. What problems or errors exist
        3. Specific improvements needed

        Critique:"""

        response = self.model.generate_content(prompt).text
        return response.strip()

    def refine(self, task, output, critique):
        """Refine output based on critique"""
        prompt = f"""Task: {task}

        Current Output:
        {output}

        Critique:
        {critique}

        Based on the critique, generate an improved version:"""

        response = self.model.generate_content(prompt).text
        return response.strip()

    def evaluate_quality(self, task, output):
        """Evaluate if output meets quality standards (0-10)"""
        prompt = f"""Task: {task}

        Output:
        {output}

        Rate the quality of this output (0-10) based on:
        - Correctness
        - Completeness
        - Clarity
        - Relevance

        Score (just number):"""

        response = self.model.generate_content(prompt).text

        try:
            score = float(response.strip().split()[0])
            return min(max(score, 0), 10)
        except:
            return 5.0

    def self_refine(self, task, max_iterations=5, quality_threshold=8.0):
        """Main self-refinement loop"""
        print(f"\n{'='*60}")
        print(f"Self-Refine Loop")
        print(f"{'='*60}")
        print(f"Task: {task}\n")

        # Step 1: Generate initial output
        print(f"{'─'*60}")
        print(f"ITERATION 1: Initial Generation")
        print(f"{'─'*60}\n")

        output = self.generate(task)
        quality = self.evaluate_quality(task, output)

        print(f"Generated Output:\n{output}\n")
        print(f"Quality Score: {quality}/10\n")

        self.history.append({
            "iteration": 1,
            "output": output,
            "quality": quality,
            "critique": None
        })

        # Refinement loop
        for iteration in range(2, max_iterations + 1):
            # Check if quality threshold met
            if quality >= quality_threshold:
                print(f"[OK] Quality threshold ({quality_threshold}) reached!")
                break

            print(f"{'─'*60}")
            print(f"ITERATION {iteration}: Critique & Refine")
            print(f"{'─'*60}\n")

            # Step 2: Self-critique
            print("Self-Critique:")
            critique = self.critique(task, output)
            print(f"{critique}\n")

            # Step 3: Refine based on critique
            print("Refining output...\n")
            output = self.refine(task, output, critique)

            # Step 4: Evaluate new quality
            quality = self.evaluate_quality(task, output)

            print(f"Refined Output:\n{output}\n")
            print(f"Quality Score: {quality}/10\n")

            self.history.append({
                "iteration": iteration,
                "output": output,
                "quality": quality,
                "critique": critique
            })

            # Check for diminishing returns
            if iteration > 2:
                prev_quality = self.history[-2]["quality"]
                improvement = quality - prev_quality
                if improvement < 0.5:
                    print(f"[WARN]  Minimal improvement ({improvement:.1f}), stopping.")
                    break

        # Show final result
        print(f"{'='*60}")
        print(f"FINAL OUTPUT (Iteration {iteration})")
        print(f"{'='*60}")
        print(output)
        print(f"\nFinal Quality: {quality}/10\n")

        self._show_history()

        return output

    def _show_history(self):
        """Display refinement history"""
        print(f"{'='*60}")
        print(f"REFINEMENT HISTORY")
        print(f"{'='*60}")

        for entry in self.history:
            print(f"Iteration {entry['iteration']}: Quality {entry['quality']:.1f}/10")

        if len(self.history) > 1:
            improvement = self.history[-1]["quality"] - self.history[0]["quality"]
            print(f"\nTotal Improvement: +{improvement:.1f} points")
        print()

In [6]:
# Example 1: Code Refinement
print("="*60)
print("EXAMPLE 1: Code Refinement")
print("="*60)

agent1 = SelfRefineAgent()
agent1.self_refine(
    "Write a Python function to find the factorial of a number",
    max_iterations=4,
    quality_threshold=8.5
)


# Example 2: Writing Improvement
print("\n" + "="*60)
print("EXAMPLE 2: Writing Improvement")
print("="*60)

agent2 = SelfRefineAgent()
agent2.self_refine(
    "Write a professional email declining a job offer politely",
    max_iterations=4,
    quality_threshold=8.0
)


# Example 3: Translation Polishing
print("\n" + "="*60)
print("EXAMPLE 3: Translation Polishing")
print("="*60)

agent3 = SelfRefineAgent()
agent3.self_refine(
    "Translate 'The early bird catches the worm' to Spanish, preserving the meaning and cultural relevance",
    max_iterations=3,
    quality_threshold=8.5
)


# Example 4: Algorithm Enhancement
print("\n" + "="*60)
print("EXAMPLE 4: Algorithm Enhancement")
print("="*60)

agent4 = SelfRefineAgent()
agent4.self_refine(
    "Design an algorithm to find the shortest path in a weighted graph",
    max_iterations=4,
    quality_threshold=8.0
)


# Example 5: Content Summarization
print("\n" + "="*60)
print("EXAMPLE 5: Content Summarization")
print("="*60)

agent5 = SelfRefineAgent()
agent5.self_refine(
    "Summarize the key benefits of remote work in 3 concise bullet points",
    max_iterations=3,
    quality_threshold=8.5
)


# Example 6: Creative Writing
print("\n" + "="*60)
print("EXAMPLE 6: Creative Idea Iteration")
print("="*60)

agent6 = SelfRefineAgent()
agent6.self_refine(
    "Create a tagline for an eco-friendly water bottle brand",
    max_iterations=4,
    quality_threshold=8.0
)


print("[OK] Self-Refine Complete!")

EXAMPLE 1: Code Refinement

Self-Refine Loop
Task: Write a Python function to find the factorial of a number

────────────────────────────────────────────────────────────
ITERATION 1: Initial Generation
────────────────────────────────────────────────────────────



Generated Output:
Here are a few ways to write a Python function to find the factorial of a number.

### 1. Iterative Approach (Recommended)
This method uses a loop, which is efficient and avoids recursion depth limits for large numbers.

```python
def factorial_iterative(n):
    """Calculates the factorial of a non-negative integer using a loop."""
    if not isinstance(n, int) or n < 0:
        raise ValueError("Factorial is only defined for non-negative integers.")

    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


# Example usage:
print(factorial_iterative(5))  # Output: 120
print(factorial_iterative(0))  # Output: 1
```

---

### 2. Recursive Approach
This method calls the function itself, matching the mathematical definition of a factorial ($n! = n \times (n-1)!$).

```python
def factorial_recursive(n):
    """Calculates the factorial of a non-negative integer using recursion."""
    if not isinstance(n, int) or n < 0:
        raise ValueError("

Generated Output:
**Subject:** Job Offer – [Position Title] – [Your Name]

Dear [Hiring Manager's Name],

Thank you very much for offering me the position of [Position Title] at [Company Name]. I sincerely appreciate the time and effort you and your team spent interviewing me and sharing more about the company's goals and culture.

After careful consideration, I have decided to decline the offer, as I have accepted another opportunity that more closely aligns with my current career path and goals.

This was a difficult decision, as I was genuinely impressed by the team and the exciting work being done at [Company Name]. 

Thank you once again for your time, consideration, and generosity throughout the hiring process. I wish you and [Company Name] continued success, and I hope our paths cross again in the future.

Sincerely,

[Your Name]  
[Your Phone Number]  
[Your LinkedIn Profile / Email Address]

Quality Score: 10.0/10

[OK] Quality threshold (8.0) reached!
FINAL OUTPUT (Iteration 

Generated Output:
**"A quien madruga, Dios le ayuda"** 
*(or "Al que madruga, Dios lo ayuda")*

**Explanation:**  
While the literal translation would be *"El pájaro tempranero atrapa el gusano"*, Spanish speakers use the traditional proverb above. It translates literally to *"God helps those who wake up early"* and carries the exact same cultural meaning: being proactive, diligent, and starting early leads to success.

Quality Score: 10.0/10

[OK] Quality threshold (8.5) reached!
FINAL OUTPUT (Iteration 2)
**"A quien madruga, Dios le ayuda"** 
*(or "Al que madruga, Dios lo ayuda")*

**Explanation:**  
While the literal translation would be *"El pájaro tempranero atrapa el gusano"*, Spanish speakers use the traditional proverb above. It translates literally to *"God helps those who wake up early"* and carries the exact same cultural meaning: being proactive, diligent, and starting early leads to success.

Final Quality: 10.0/10

REFINEMENT HISTORY
Iteration 1: Quality 10.0/10


EXAMPLE

Generated Output:
Here is a comprehensive design for finding the single-source shortest path in a weighted graph using **Dijkstra’s Algorithm** (with a Min-Heap / Priority Queue), assuming edge weights are non-negative.

---

### 1. Algorithm Overview: Dijkstra's Algorithm

Dijkstra’s algorithm uses a **greedy approach** to iteratively find the shortest path from a starting node (source) to all other nodes in a graph.

#### Key Concepts:
- **Distance Table (`dist`)**: Keeps track of the minimum known distance from the source to every node.
- **Priority Queue / Min-Heap (`pq`)**: Efficiently retrieves the unvisited node with the smallest tentative distance.
- **Predecessor Map (`parent`)**: Stores the previous node in the shortest path to reconstruct the actual path once the destination is reached.
- **Edge Relaxation**: The process of updating the shortest distance to a neighbor $v$ via node $u$ if:
  $$\text{dist}[u] + \text{weight}(u, v) < \text{dist}[v]$$

---

### 2. Step-by-Step A

Generated Output:
* **Improved Work-Life Balance:** Eliminates daily commute times and offers schedule flexibility, leading to reduced stress and more personal time.
* **Cost Savings:** Lowers transportation, food, and wardrobe expenses for employees while significantly reducing overhead and real estate costs for employers.
* **Broader Talent Access and Productivity:** Enables companies to hire from a global talent pool while allowing employees to work in personalized, distraction-free environments.

Quality Score: 10.0/10

[OK] Quality threshold (8.5) reached!
FINAL OUTPUT (Iteration 2)
* **Improved Work-Life Balance:** Eliminates daily commute times and offers schedule flexibility, leading to reduced stress and more personal time.
* **Cost Savings:** Lowers transportation, food, and wardrobe expenses for employees while significantly reducing overhead and real estate costs for employers.
* **Broader Talent Access and Productivity:** Enables companies to hire from a global talent pool

Generated Output:
**"Refill your body. Restore the planet."**

---

*Alternative options based on brand tone:*

* **Direct & Impactful:** *Pure hydration. Zero waste.*
* **Action-Oriented:** *Choose to reuse. Sip for the planet.*
* **Minimalist:** *Hydrate consciously.*
* **Inspiring:** *One bottle today, a greener Earth tomorrow.*

Quality Score: 10.0/10

[OK] Quality threshold (8.0) reached!
FINAL OUTPUT (Iteration 2)
**"Refill your body. Restore the planet."**

---

*Alternative options based on brand tone:*

* **Direct & Impactful:** *Pure hydration. Zero waste.*
* **Action-Oriented:** *Choose to reuse. Sip for the planet.*
* **Minimalist:** *Hydrate consciously.*
* **Inspiring:** *One bottle today, a greener Earth tomorrow.*

Final Quality: 10.0/10

REFINEMENT HISTORY
Iteration 1: Quality 10.0/10

[OK] Self-Refine Complete!
